# Explainability — Grad-CAM and Perona-Malik grid maps

This notebook visualises **why** the federated ResNet18 predicts each class.

- **Grad-CAM** (target: `layer4`, final residual stage) highlights image regions
  that most influenced the classifier logit for the predicted class.
- **Perona-Malik grid map** uses the **same `feature_layer` and `grid_size`** as
  PIDL training (default `layer2`, grid 4×4). Each cell score is the mean squared
  PM residual in that region — strong values indicate pronounced edge/texture
  structure under the physics-inspired operator.
- **Overlap (IoU, top 25%)** compares where Grad-CAM and the PM map agree.
  Agreement suggests predictions align with physics-guided spatial features.

**Modes:** if `final_model.pth` exists under `results_explainability/...`, weights
are loaded; otherwise the full Flower FL pipeline is re-run and the global model
is saved.


---
## A. Setup

Optional Google Drive mount (skip if you store the repo only under `/content`).


In [ ]:
# Optional: Google Drive (uncomment if you keep data or repo on Drive)
# from google.colab import drive
# drive.mount('/content/drive')

import os, sys, json, gc, time
from pathlib import Path

GITHUB_REPO = 'https://github.com/YOUR_USERNAME/medical_fl_pidl.git'
PROJECT_ROOT = Path('/content/medical_fl_pidl')

if not PROJECT_ROOT.exists():
    os.system(f'git clone {GITHUB_REPO} {PROJECT_ROOT}')
    if not PROJECT_ROOT.exists():
        raise RuntimeError('Clone failed — set GITHUB_REPO to your fork.')
else:
    os.system(f'git -C {PROJECT_ROOT} pull')

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

!pip install -q --upgrade pip setuptools wheel
!pip install -q -r requirements.txt

import torch
print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())


---
## A (continued). Experiment configuration

Edit `dataset_name` and `KAGGLE_SLUGS` mapping as needed. FL hyperparameters should
match training you want to explain.


In [ ]:
from __future__ import annotations

import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# ── Dataset selection (edit slug in one place) ────────────────────
dataset_name = 'brain_tumor_mri'   # brain_tumor_mri | colon_cancer_or_pathology | covid

KAGGLE_SLUGS = {
    'brain_tumor_mri':          'masoudnickparvar/brain-tumor-mri-dataset',
    'colon_cancer_or_pathology': 'andrewmvd/lung-and-colon-cancer-histopathological-images',
    'covid':                    'tawsifurrahman/covid19-radiography-database',
}
COLON_OR_LUNG = 'colon_image_sets'  # for colon dataset only

# ── Federated training (match main experiments) ───────────────────
num_clients     = 3
num_rounds      = 5
local_epochs    = 2
batch_size      = 32
learning_rate   = 1e-3
image_size      = 224
random_seed     = 42
feature_layer   = 'layer2'
grid_size       = 4
lambda_pm       = 0.10
regularizer_type = 'perona_malik'
use_grid_loss   = True
k_pm            = 1.0
use_secagg      = False   # SecAgg+ off in run_simulation (same as notebooks 01/04)

# ── Explainability ────────────────────────────────────────────────
samples_per_class = 5
gradcam_layer     = 'layer4'   # final conv stage for Grad-CAM

RESULT_DIR = PROJECT_ROOT / 'results_explainability' / dataset_name / f'{num_clients}_clients'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
FIG = RESULT_DIR / 'figures'
(FIG / 'gradcam').mkdir(parents=True, exist_ok=True)
(FIG / 'pm_grid').mkdir(parents=True, exist_ok=True)
(FIG / 'combined').mkdir(parents=True, exist_ok=True)
(FIG / 'summary').mkdir(parents=True, exist_ok=True)

print('Results ->', RESULT_DIR)


---
## B. Download data and resolve ImageFolder root


In [ ]:
import kagglehub
from data.kaggle_loader import find_image_root

slug = KAGGLE_SLUGS[dataset_name]
raw = kagglehub.dataset_download(slug)
print('Downloaded:', raw)

if dataset_name == 'colon_cancer_or_pathology':
    cand = list(Path(raw).rglob(COLON_OR_LUNG))
    data_root = str(cand[0]) if cand else str(find_image_root(raw))
else:
    data_root = str(find_image_root(raw))

print('image_root:', data_root)


---
## C. Model handling — load `final_model.pth` or retrain FL


In [ ]:
model_path = RESULT_DIR / 'final_model.pth'
need_train = not model_path.exists()
print('final_model.pth exists:', not need_train)

if need_train:
    import json as _json
    import gc as _gc
    from flwr.simulation import run_simulation
    from federated.server_app import app as _server_app
    from federated.client_app import _make_client_app

    run_cfg = {
        'dataset_name': dataset_name,
        'data_root': data_root,
        'log_dir': str(RESULT_DIR),
        'num_classes': 0,
        'num_clients': num_clients,
        'num_server_rounds': num_rounds,
        'min_fit_clients': num_clients,
        'local_epochs': local_epochs,
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'image_size': image_size,
        'feature_layer': feature_layer,
        'regularizer_type': regularizer_type,
        'lambda_pm': lambda_pm,
        'use_grid_loss': use_grid_loss,
        'grid_size': grid_size,
        'k': k_pm,
        'random_seed': random_seed,
        'use_secagg': use_secagg,
        'enable_attack': False,
        'enable_update_clipping': False,
    }
    os.environ['FL_RUN_OVERRIDE'] = _json.dumps(run_cfg)
    _client_app = _make_client_app()
    gpu_frac = 0.5 if torch.cuda.is_available() else 0.0
    t0 = time.time()
    run_simulation(
        server_app=_server_app,
        client_app=_client_app,
        num_supernodes=num_clients,
        backend_config={'client_resources': {'num_cpus': 2, 'num_gpus': gpu_frac}},
    )
    print(f'FL done in {time.time()-t0:.0f}s')
    try:
        from federated.server_app import finalize_experiment
        finalize_experiment()
    except Exception as exc:
        print('finalize_experiment:', exc)
    os.environ.pop('FL_RUN_OVERRIDE', None)
    _gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

assert model_path.exists(), 'final_model.pth missing after training'
print('Using weights:', model_path)


---
## D. Build global test loader & select balanced samples


In [ ]:
from federated.task import get_federated_data
from configs.experiment_config import ModelConfig
from models.resnet_pidl import build_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

data = get_federated_data(
    data_root=data_root,
    num_clients=num_clients,
    batch_size=batch_size,
    image_size=image_size,
    random_seed=random_seed,
    save_summary_to=str(RESULT_DIR),
)
num_classes = data['num_classes']
class_names = list(data['class_names'])
test_loader = data['global_test_loader']

# Navigate: TransformDataset -> Subset -> ImageFolder
tds = test_loader.dataset
sub = tds.subset
base_folder = sub.dataset
subset_indices = list(sub.indices)

by_class = {c: [] for c in range(num_classes)}
for j in range(len(sub)):
    idx = subset_indices[j]
    y = int(base_folder.samples[idx][1])
    by_class[y].append(j)

rng = random.Random(random_seed)
picked = []
for c in range(num_classes):
    pool = by_class[c][:]
    rng.shuffle(pool)
    for j in pool[:samples_per_class]:
        idx = subset_indices[j]
        path = base_folder.samples[idx][0]
        picked.append({'ds_index': j, 'true_class': c, 'class_name': class_names[c], 'path': path})

pd.DataFrame(picked).to_csv(RESULT_DIR / 'selected_samples.csv', index=False)
print(f'Selected {len(picked)} samples')


---
## E–G. Run Grad-CAM, PM grid maps, combined figures, CSV log


In [ ]:
import matplotlib.pyplot as plt

from explainability.gradcam import GradCAM, cam_statistics, upsample_cam
from explainability.pm_grid_explainer import (
    gradcam_pm_iou_top25,
    grid_statistics,
    normalize01,
    pm_grid_scores,
    upsample_grid_map,
)
from explainability.plot_utils import overlay_heatmap, savefig_tight, tensor_to_display_rgb

model_cfg = ModelConfig(pidl_feature_layer=feature_layer)  # type: ignore[arg-type]
model = build_model(num_classes=num_classes, config=model_cfg)
sd = torch.load(model_path, map_location='cpu')
model.load_state_dict(sd)
model.to(device)
model.eval()

target_mod = getattr(model, gradcam_layer)

rows = []
y_true_sel = []
y_pred_sel = []

for i, rec in enumerate(picked):
    image_id = f'{i:04d}'
    x, _ = tds[rec['ds_index']]
    x = x.unsqueeze(0).to(device)
    H = W = image_size

    with GradCAM(model, target_mod) as gcam:
        cam_hw, logits, pred_idx = gcam.compute(x, class_idx=None)

    prob = torch.softmax(logits, dim=-1)[0, pred_idx].item()
    true_idx = rec['true_class']
    ok = pred_idx == true_idx
    y_true_sel.append(true_idx)
    y_pred_sel.append(pred_idx)

    cam_up = upsample_cam(cam_hw, H, W)
    mean_gc, max_gc = cam_statistics(cam_hw)
    model.zero_grad(set_to_none=True)

    with torch.no_grad():
        _, feats = model(x, return_features=True)
        fm_pidl = feats[feature_layer]

    gs = pm_grid_scores(fm_pidl, grid_size=grid_size, k=k_pm, normalize=True)
    gs_n = normalize01(gs)[0]
    pm_up = upsample_grid_map(gs_n.unsqueeze(0), H, W)
    mean_pm, max_pm, top_cell = grid_statistics(gs_n.unsqueeze(0))

    iou_25 = gradcam_pm_iou_top25(cam_up, pm_up)

    rgb = tensor_to_display_rgb(x[0], (H, W))
    o_gc = overlay_heatmap(rgb, cam_up)
    o_pm = overlay_heatmap(rgb, pm_up, cmap='viridis')

    title = (
        f"true={rec['class_name']} | pred={class_names[pred_idx]} | conf={prob:.3f} | ok={ok}"
    )

    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    ax.imshow(o_gc)
    ax.set_title('Grad-CAM\n' + title, fontsize=8)
    ax.axis('off')
    savefig_tight(FIG / 'gradcam' / f'{image_id}_gradcam.png')

    fig, ax = plt.subplots(1, 1, figsize=(4, 4))
    ax.imshow(o_pm)
    ax.set_title('PM grid\n' + title, fontsize=8)
    ax.axis('off')
    savefig_tight(FIG / 'pm_grid' / f'{image_id}_pmgrid.png')

    fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
    axes[0].imshow(rgb)
    axes[0].set_title('Input')
    axes[1].imshow(o_gc)
    axes[1].set_title('Grad-CAM')
    axes[2].imshow(o_pm)
    axes[2].set_title('PM grid')
    axes[3].imshow(np.concatenate([o_gc, o_pm], axis=1))
    axes[3].set_title('GC | PM')
    for ax in axes:
        ax.axis('off')
    fig.suptitle(title, fontsize=9)
    savefig_tight(FIG / 'combined' / f'{image_id}_combined.png')

    rows.append({
        'dataset_name': dataset_name,
        'num_clients': num_clients,
        'image_id': image_id,
        'true_class': rec['class_name'],
        'predicted_class': class_names[pred_idx],
        'confidence': round(prob, 6),
        'correct': ok,
        'mean_gradcam_score': round(mean_gc, 6),
        'max_gradcam_score': round(max_gc, 6),
        'mean_pm_grid_score': round(mean_pm, 6),
        'max_pm_grid_score': round(max_pm, 6),
        'top_pm_grid_index': top_cell,
        'gradcam_pm_overlap_score': round(iou_25, 6),
    })

summary_df = pd.DataFrame(rows)
summary_df.to_csv(RESULT_DIR / 'explainability_summary.csv', index=False)
print(summary_df.head())


---
## H. Save config snapshot


In [ ]:
cfg_out = {
    'dataset_name': dataset_name,
    'data_root': data_root,
    'num_clients': num_clients,
    'num_rounds': num_rounds,
    'local_epochs': local_epochs,
    'batch_size': batch_size,
    'image_size': image_size,
    'feature_layer': feature_layer,
    'grid_size': grid_size,
    'lambda_pm': lambda_pm,
    'k_pm': k_pm,
    'gradcam_layer': gradcam_layer,
    'samples_per_class': samples_per_class,
    'random_seed': random_seed,
}
(RESULT_DIR / 'explainability_config.json').write_text(json.dumps(cfg_out, indent=2))


---
## I. Summary plots and tables


In [ ]:
# Mean overlap & confidence by true class
by_cls = summary_df.groupby('true_class').agg(
    gradcam_pm_overlap=('gradcam_pm_overlap_score', 'mean'),
    confidence=('confidence', 'mean'),
).reset_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(by_cls['true_class'], by_cls['gradcam_pm_overlap'], color='steelblue')
ax.set_ylabel('Mean Grad-CAM / PM IoU (top 25%)')
ax.set_title('Overlap by true class')
plt.xticks(rotation=30, ha='right')
savefig_tight(FIG / 'summary' / 'bar_overlap_by_class.png')

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(by_cls['true_class'], by_cls['confidence'], color='darkseagreen')
ax.set_ylabel('Mean confidence')
ax.set_title('Confidence by true class (selected samples)')
plt.xticks(rotation=30, ha='right')
savefig_tight(FIG / 'summary' / 'bar_confidence_by_class.png')

from IPython.display import display
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true_sel, y_pred_sel, labels=list(range(num_classes)))
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(num_classes))
ax.set_yticks(range(num_classes))
ax.set_xticklabels(class_names, rotation=30, ha='right')
ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
fig.colorbar(im, ax=ax)
fig.suptitle('Confusion — selected samples only')
savefig_tight(FIG / 'summary' / 'confusion_selected.png')

correct_tbl = summary_df.groupby('correct').size().rename('count').reset_index()
correct_tbl.to_csv(RESULT_DIR / 'explanation_correctness_table.csv', index=False)
display(correct_tbl)


---
## Download (Colab)


In [ ]:
import subprocess
zip_path = '/content/explainability_results.zip'
subprocess.run(['zip', '-r', zip_path, str(RESULT_DIR)], check=True)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Not on Colab — results in', RESULT_DIR)
